In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader, MatchedSiphraAcquisition, match_events
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

colors = [
    ROOT.kRed,
    ROOT.kBlue,
    ROOT.kGreen + 2,
    ROOT.kOrange + 7,
    ROOT.kMagenta,
    ROOT.kCyan + 2,
    ROOT.kViolet,
    ROOT.kAzure + 7,
    ROOT.kPink + 7,
    ROOT.kTeal + 3,
    ROOT.kSpring + 5,
    ROOT.kYellow + 2,
    ROOT.kOrange + 1,
    ROOT.kBlue - 7,
    ROOT.kRed - 7,
    ROOT.kGray + 2,
    ROOT.kBlack,
    ROOT.kSpring - 7
]

In [2]:
siphras = ['A', 'B']

files_A = sorted((project_root/'data/261012').glob('SUBT_A_*.csv'))
files_B = sorted((project_root/'data/261012').glob('SUBT_B_*.csv'))

acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]

acqs_AB = [acqs_A, acqs_B]

print(acqs_AB)

[[SIPHRAAcquisition(File: 'SUBT_A_01_18ChTest_BG'), SIPHRAAcquisition(File: 'SUBT_A_02_18ChTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_A_03_18ChTest_Co60'), SIPHRAAcquisition(File: 'SUBT_A_04_18ChTest_Na22')], [SIPHRAAcquisition(File: 'SUBT_B_01_18ChTest_BG'), SIPHRAAcquisition(File: 'SUBT_B_02_18ChTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_B_03_18ChTest_Co60'), SIPHRAAcquisition(File: 'SUBT_B_04_18ChTest_Na22')]]


In [3]:
#histogram
hists = {}

sources = ['Cs-137', 'Na-22', 'Co-60']
    
channels = ['Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8', 'Ch9', 'Ch10', 'Ch11', 'Ch12', 'Ch13', 'Ch14',
            'Ch15', 'Ch16', 'Siphra_sum', 'Manual_sum']


for j, siphra in enumerate(siphras):

    acqs = acqs_AB[j]
    print(acqs)

    
    for i, source in enumerate(sources):
        bg = acqs[0]
        sgnl = acqs[i+1]
        for channel in channels:
            hist_sgnl = ROOT.TH1F(f"{channel}_{source}_SIPHRA_{siphra}_signal", "", BITS12, 0, BITS12)
            hist_bg = ROOT.TH1F(f"{channel}_{source}_SIPHRA_{siphra}_BG", "", BITS12, 0, BITS12)
            if channel == "Manual_sum":
                hist_sgnl.Fill(sgnl[channel]/16)
                hist_bg.Fill(bg[channel]/16)
            else: 
                hist_sgnl.Fill(sgnl[channel])
                hist_bg.Fill(bg[channel])
    
            hist_bg.Scale(sgnl.exposure/bg.exposure)
    
            hist_clean = hist_sgnl.Clone(f"{channel}_{source}_clean")
            hist_clean.Add(hist_bg, -1)
    
            for hist in [hist_sgnl, hist_bg, hist_clean]:
                # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
                hist.GetXaxis().SetTitle("ADC channel number")
                hist.GetYaxis().SetTitle(r"Normalized counts")
    
            hists[f"{channel}_{source}_SIPHRA_{siphra}_signal"] = hist_sgnl
            hists[f"{channel}_{source}_SIPHRA_{siphra}_clean"] = hist_clean
            hists[f"{channel}_{source}_SIPHRA_{siphra}_BG"] = hist_bg
            del hist_sgnl, hist_clean, hist_bg


[SIPHRAAcquisition(File: 'SUBT_A_01_18ChTest_BG'), SIPHRAAcquisition(File: 'SUBT_A_02_18ChTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_A_03_18ChTest_Co60'), SIPHRAAcquisition(File: 'SUBT_A_04_18ChTest_Na22')]
[SIPHRAAcquisition(File: 'SUBT_B_01_18ChTest_BG'), SIPHRAAcquisition(File: 'SUBT_B_02_18ChTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_B_03_18ChTest_Co60'), SIPHRAAcquisition(File: 'SUBT_B_04_18ChTest_Na22')]


In [4]:
channels = ["Siphra_sum", "Manual_sum"]


if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 2100, 1500)
cv.Divide(3, 2)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(9)]

for n, siphra in enumerate(siphras):
    cv.cd(n*3+i+1)
    cv.cd(n*3+i+1).SetLogy()
    lg = lgds[n*3+i]
    for j, channel in enumerate(channels):
        hist = hists[f"{channel}_{source}_SIPHRA_{siphra}_clean"]
        lg.AddEntry(hist, channel)
        hist.SetLineColor(colors[j+1])
        if j == 0:
            hist.Draw("hist")
        else:
            hist.Draw("sames hist")
        ROOT.gPad.Update()
        
        
    lg.Draw()
    cv.Draw()

    
    for i, source in enumerate(sources):
        print(siphra)
        cv.cd(n*3+i+1)
        cv.cd(n*3+i+1).SetLogy()
        lg = lgds[n*3+i]
        for j, channel in enumerate(channels):
            hist = hists[f"{channel}_{source}_SIPHRA_{siphra}_clean"]
            lg.AddEntry(hist, channel)
            hist.SetLineColor(colors[j+1])
            if j == 0:
                hist.Draw("hist")
            else:
                hist.Draw("sames hist")
            ROOT.gPad.Update()
            
            
        lg.Draw()
        cv.Draw()

#print(hists)

A
A
A
B
B
B


In [5]:
from analysis import *


sources = ['Cs-137', 'Na-22', 'Co-60']
channels = ["Siphra_sum", "Manual_sum"]

energies = {}
energies['Cs-137'] = CS137_MEV
energies['Na-22'] = NA22_MEV
energies['Co-60'] = CO60_MEV

ranges_ss = {}
ranges_ss["Co-60"] = [[350, 500], [770, 910]]
ranges_ss['Na-22'] = [[110, 200], [370, 500], [530, 640]]
ranges_ss['Cs-137'] = [[100, 300]]

ranges_ms = {}
ranges_ms["Co-60"] = [[200, 360], [690, 820]]
ranges_ms['Na-22'] = [[110, 210], [360, 460], [490, 590]]
ranges_ms['Cs-137'] = [[100, 300]]

ranges = {}
ranges['Siphra_sum'] = ranges_ss
ranges['Manual_sum'] = ranges_ms

energy_vs_diff = []
resolutions = [ [],[] ]

for siphra in siphras:
    for source in sources:
        means = []
        for i, channel in enumerate(channels):
            rng = ranges[channel][source]
            hist = hists[f"{channel}_{source}_SIPHRA_{siphra}_clean"]
            means.append([])
            #resolutions.append([])
    
            
            
            for j, r in enumerate(rng):
                gauss = ROOT.TF1(f"gauss_{channel}_{source}_SIPHRA_{siphra}_clean", "gaus", r[0], r[1])
                hist.Fit(gauss, "R+S")
                mean = gauss.GetParameter(1)
                means[i].append(mean)
                resolution = energy_resolution(hist, [r])
                print(f"Energy resolution SIPHRA {siphra} {source} {energies[source][j]:.4g} MeV {channel}: {resolution[0]*100:.3g} %" )
                resolutions[i].append((energies[source][j], resolution[0]*100))
    
        for m in range(len(means[0])):
            diff = means[0][m]/means[1][m]
            diff2 = (means[0][m]-means[1][m])/means[0][m]
            #print(diff2)
            #print(f"{source} (energy: {energies[source][m]:.4g} MeV), diff: {(diff2)*100:.3g} %") 
            energy_vs_diff.append((energies[source][m], diff2*100))

#print(energy_vs_diff)
print(resolutions)


Energy resolution SIPHRA A Cs-137 0.661 MeV Siphra_sum: 29.6 %
Energy resolution SIPHRA A Cs-137 0.661 MeV Manual_sum: 58 %


ZeroDivisionError: division by zero

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      13777.9
NDf                       =          197
Edm                       =   3.0802e-08
NCalls                    =          115
Constant                  =      647.029   +/-   10.1943     
Mean                      =      301.374   +/-   1.26685     
Sigma                     =      37.9549   +/-   0.576659     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      13777.9
NDf                       =          197
Edm                       =   3.0802e-08
NCalls                    =          115
Constant                  =      647.029   +/-   10.1943     
Mean                      =      301.374   +/-   1.26685     
Sigma                     =      37.9549   +/-   0.576659     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      866.698
NDf                   

Warning in <Fit>: Abnormal termination of minimization.
Warning in <Fit>: Abnormal termination of minimization.
Warning in <TFitResult::GetCovarianceMatrix>: covariance matrix is not available
Error in <TMatrixTRow_const(const TMatrixTSym &,Int_t)>: row index out of bounds


In [ ]:
import matplotlib.pyplot as plt

#data = [(1, 2), (2, 4), (3, 3), (4, 7), (5, 6)]

x, y = zip(*energy_vs_diff)

m, b = np.polyfit(x, y, 1)

plt.scatter(x, y, marker="o")

plt.plot(x, np.array(x) * m + b, label=f"fit: y = {m:.2f}x + {b:.2f}")

plt.xlabel("Energy [MeV]")
plt.ylabel("Difference [%]")
plt.show()



In [ ]:
import matplotlib.pyplot as plt

for i in range(2):

    data = resolutions[i]
    channel = channels[i]
    x, y = zip(*data)
    m, b = np.polyfit(x, y, 1)
    
    plt.scatter(x, y, marker="o")
    
    plt.plot(x, np.array(x) * m + b, label=f"{channel}")
    
    plt.xlabel("Energy [MeV]")
    plt.ylabel("Resolution [%]")
plt.legend()    
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def exponential(x, A, B):
    return A * np.exp(B * x)

for i in range(2):

    channel = channels[i]

    data = resolutions[i]
    x, y = zip(*data)

    # Fit exponential
    popt, pcov = curve_fit(exponential, x, y)
    A, B = popt

    # Scatter data
    plt.scatter(x, y, marker="o")

    # Smooth fitted curve
    x_fit = np.linspace(min(x), max(x), 200)
    y_fit = exponential(x_fit, A, B)

    plt.plot(
        x_fit,
        y_fit,
        label=f"{channel}"
    )

    plt.xlabel("Energy [MeV]")
    plt.ylabel("Resolution [%]")

plt.legend()
plt.show()

In [ ]:
data = acqs[3]
print(data)

ms = data['Manual_sum']
ss = data['Siphra_sum']

plt.scatter(ms, ss, marker='o')

# Linear fit
m, b = np.polyfit(ms, ss, 1)

# Predicted values from the fit
ss_fit = m * np.array(ms) + b

# Calculate R^2
ss_res = np.sum((ss - ss_fit)**2)
ss_tot = np.sum((ss - np.mean(ss))**2)
r_squared = 1 - ss_res / ss_tot

# Plot
plt.plot(
    ms,
    ss_fit,
    label=f"fit: y = {m:.2f}x + {b:.2f}\n$R^2$ = {r_squared:.5f}"
)

plt.ylabel("Siphra sum")
plt.xlabel("Manual sum")

plt.legend()
plt.show()

print("Slope:", m)
print("Intercept:", b)
print("Slope × 16:", m * 16)
print("R²:", r_squared)
